# V6-1 `credit_bureau_a_2` Two-Stage Feature-Only OOM Test

This notebook intentionally does not train or load a model.

It tests whether `credit_bureau_a_2` can be aggregated as:

```text
payment rows -> case_id + num_group1 contract rows -> case_id rows
```

The final submission uses a constant `0.5` score.

In [ ]:
from __future__ import annotations

import gc
import os
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")

KAGGLE_ROOT_CANDIDATES = [
    Path("/kaggle/input/home-credit-credit-risk-model-stability"),
    Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability"),
]


def find_local_data_dir() -> Path:
    candidates = [Path("data"), Path.cwd() / "data"]
    candidates.extend(parent / "data" for parent in Path.cwd().resolve().parents)
    for candidate in candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find competition data.")


def find_kaggle_data_dir() -> Path | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for sample_path in input_root.rglob("sample_submission.csv"):
        data_dir = sample_path.parent
        if (data_dir / "parquet_files" / "test" / "test_base.parquet").exists():
            return data_dir
    return None


KAGGLE_ROOT = next((path for path in KAGGLE_ROOT_CANDIDATES if path.exists()), None)
ROOT = KAGGLE_ROOT if KAGGLE_ROOT is not None else (find_kaggle_data_dir() or find_local_data_dir())
TEST_DIR = ROOT / "parquet_files" / "test"
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/v6_a2_twostage_feature_only")
WORKING.mkdir(parents=True, exist_ok=True)

MAX_RSS_GB = 30.0
MIN_AVAILABLE_GB = 8.0
A2_SELECTED_COLUMNS = [
    "pmts_dpd_1073P",
    "pmts_dpd_303P",
    "pmts_overdue_1140A",
    "pmts_overdue_1152A",
    "pmts_month_158T",
    "pmts_month_706T",
    "pmts_year_1139T",
    "pmts_year_507T",
]
OVERDUE_COLUMNS = [
    "pmts_dpd_1073P",
    "pmts_dpd_303P",
    "pmts_overdue_1140A",
    "pmts_overdue_1152A",
]

print({
    "root": str(ROOT),
    "test_dir": str(TEST_DIR),
    "working": str(WORKING),
})

In [ ]:
class MemoryLimitExceeded(RuntimeError):
    pass


def _process_memory_mb() -> float | None:
    try:
        import psutil

        return psutil.Process(os.getpid()).memory_info().rss / 1024**2
    except Exception:
        pass

    if sys.platform.startswith("linux") or sys.platform == "darwin":
        try:
            import resource

            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            return rss / 1024 if sys.platform.startswith("linux") else rss / 1024**2
        except Exception:
            return None
    return None


def _available_memory_gb() -> float | None:
    try:
        import psutil

        return psutil.virtual_memory().available / 1024**3
    except Exception:
        return None


def log_memory(label: str) -> None:
    rss = _process_memory_mb()
    available = _available_memory_gb()
    parts = [f"[memory] {label}"]
    if rss is not None:
        parts.append(f"rss={rss:.1f} MB")
    if available is not None:
        parts.append(f"available={available:.1f} GB")
    print(" | ".join(parts))


def check_memory(label: str) -> None:
    rss_mb = _process_memory_mb()
    available_gb = _available_memory_gb()
    log_memory(label)

    if rss_mb is not None and rss_mb / 1024 > MAX_RSS_GB:
        raise MemoryLimitExceeded(
            f"Memory guard stopped at {label}: RSS {rss_mb / 1024:.1f} GB > {MAX_RSS_GB:.1f} GB"
        )
    if available_gb is not None and available_gb < MIN_AVAILABLE_GB:
        raise MemoryLimitExceeded(
            f"Memory guard stopped at {label}: available {available_gb:.1f} GB < {MIN_AVAILABLE_GB:.1f} GB"
        )


check_memory("initialized")

In [ ]:
def normalize_date_columns(lf: pl.LazyFrame) -> pl.LazyFrame:
    schema = lf.collect_schema()
    exprs = []
    for name, dtype in schema.items():
        if name.endswith("D") or name == "date_decision":
            if dtype.is_numeric():
                continue
            exprs.append(
                pl.col(name)
                .cast(pl.Utf8)
                .str.strptime(pl.Date, strict=False)
                .cast(pl.Int32)
                .alias(name)
            )
    return lf.with_columns(exprs) if exprs else lf


def _safe_div(num: pl.Expr, den: pl.Expr) -> pl.Expr:
    return pl.when(den > 0).then(num / den).otherwise(None)


def aggregate_file_to_contracts(path: Path, case_ids: pl.DataFrame, out_path: Path) -> Path:
    print(f"[a2-contract] start {path.name}")
    check_memory(f"before contract aggregate {path.name}")

    scan = pl.scan_parquet(str(path))
    schema = scan.collect_schema()
    available = [col for col in A2_SELECTED_COLUMNS if col in schema]
    if not available:
        raise ValueError(f"No selected A2 columns found in {path}")

    lf = scan.select(["case_id", "num_group1", "num_group2", *available])
    lf = lf.with_columns(
        [
            pl.col("case_id").cast(pl.Int64),
            pl.col("num_group1").cast(pl.Int64, strict=False),
            pl.col("num_group2").cast(pl.Int64, strict=False),
        ]
    )
    lf = lf.join(case_ids.lazy(), on="case_id", how="semi")

    aggs = [
        pl.len().alias("a2_contract__payment_count"),
        pl.col("num_group2").count().alias("a2_contract__num_group2_count"),
        pl.col("num_group2").max().alias("a2_contract__num_group2_max"),
        pl.col("num_group2").min().alias("a2_contract__num_group2_min"),
    ]
    overdue_flags = []
    for col in available:
        base = pl.col(col).cast(pl.Float64, strict=False)
        prefix = f"a2_contract__{col}"
        aggs.extend(
            [
                base.sum().alias(f"{prefix}__sum"),
                base.count().alias(f"{prefix}__count"),
                base.mean().alias(f"{prefix}__mean"),
                base.max().alias(f"{prefix}__max"),
                base.min().alias(f"{prefix}__min"),
                base.std().alias(f"{prefix}__std"),
                (base.max() - base.min()).alias(f"{prefix}__range"),
            ]
        )
        if col in OVERDUE_COLUMNS:
            overdue_flags.append(base.fill_null(0) > 0)
    if overdue_flags:
        aggs.append(pl.any_horizontal(overdue_flags).max().cast(pl.Int8).alias("a2_contract__has_overdue"))
    else:
        aggs.append(pl.lit(None).cast(pl.Int8).alias("a2_contract__has_overdue"))

    contracts = lf.group_by(["case_id", "num_group1"]).agg(aggs).collect(engine="streaming")
    contracts.write_parquet(out_path)
    print(f"[a2-contract] wrote {out_path.name}: {contracts.shape}")

    del contracts, lf, scan
    gc.collect()
    check_memory(f"after contract aggregate {path.name}")
    return out_path


def aggregate_contracts_to_case(contract_path: Path, out_path: Path) -> Path:
    print(f"[a2-case] start {contract_path.name}")
    check_memory(f"before case aggregate {contract_path.name}")

    lf = pl.scan_parquet(str(contract_path))
    schema = lf.collect_schema()
    aggs = [
        pl.len().alias("a2_case_partial__contract_count"),
        pl.col("a2_contract__payment_count").sum().alias("a2_case_partial__payment_count_sum"),
        pl.col("a2_contract__payment_count").max().alias("a2_case_partial__payment_count_max"),
        pl.col("a2_contract__payment_count").mean().alias("a2_case_partial__payment_count_mean"),
        pl.col("a2_contract__has_overdue").sum().alias("a2_case_partial__overdue_contract_count"),
    ]

    for col in A2_SELECTED_COLUMNS:
        prefix = f"a2_contract__{col}"
        sum_col = f"{prefix}__sum"
        count_col = f"{prefix}__count"
        max_col = f"{prefix}__max"
        min_col = f"{prefix}__min"
        mean_col = f"{prefix}__mean"
        std_col = f"{prefix}__std"
        range_col = f"{prefix}__range"
        out = f"credit_bureau_a_2__{col}"
        if sum_col not in schema:
            continue
        aggs.extend(
            [
                pl.col(sum_col).sum().alias(f"{out}__sum_sum"),
                pl.col(count_col).sum().alias(f"{out}__count_sum"),
                pl.col(max_col).max().alias(f"{out}__contract_max"),
                pl.col(min_col).min().alias(f"{out}__contract_min"),
                pl.col(mean_col).mean().alias(f"{out}__contract_mean_mean"),
                pl.col(mean_col).max().alias(f"{out}__contract_mean_max"),
                pl.col(std_col).mean().alias(f"{out}__contract_std_mean"),
                pl.col(range_col).max().alias(f"{out}__contract_range_max"),
            ]
        )

    case_partial = lf.group_by("case_id").agg(aggs).collect(engine="streaming")
    case_partial.write_parquet(out_path)
    print(f"[a2-case] wrote {out_path.name}: {case_partial.shape}")

    del case_partial, lf
    gc.collect()
    check_memory(f"after case aggregate {contract_path.name}")
    return out_path

In [ ]:
def combine_case_partials(case_paths: list[Path]) -> pl.DataFrame:
    if not case_paths:
        raise FileNotFoundError("No case partial parquet files were generated.")

    lf = pl.scan_parquet([str(path) for path in case_paths])
    schema = lf.collect_schema()
    aggs = [
        pl.col("a2_case_partial__contract_count").sum().alias("credit_bureau_a_2__contract_count"),
        pl.col("a2_case_partial__payment_count_sum").sum().alias("credit_bureau_a_2__payment_count"),
        pl.col("a2_case_partial__payment_count_max").max().alias("credit_bureau_a_2__payment_count_contract_max"),
        pl.col("a2_case_partial__payment_count_mean").mean().alias("credit_bureau_a_2__payment_count_contract_mean"),
        pl.col("a2_case_partial__overdue_contract_count").sum().alias("credit_bureau_a_2__overdue_contract_count"),
    ]

    for col in A2_SELECTED_COLUMNS:
        out = f"credit_bureau_a_2__{col}"
        sum_col = f"{out}__sum_sum"
        count_col = f"{out}__count_sum"
        max_col = f"{out}__contract_max"
        min_col = f"{out}__contract_min"
        mean_mean_col = f"{out}__contract_mean_mean"
        mean_max_col = f"{out}__contract_mean_max"
        std_mean_col = f"{out}__contract_std_mean"
        range_max_col = f"{out}__contract_range_max"
        if sum_col not in schema:
            continue
        count_sum = pl.col(count_col).sum()
        aggs.extend(
            [
                pl.col(sum_col).sum().alias(f"{out}__sum"),
                count_sum.alias(f"{out}__count"),
                _safe_div(pl.col(sum_col).sum(), count_sum).alias(f"{out}__mean"),
                pl.col(max_col).max().alias(f"{out}__max"),
                pl.col(min_col).min().alias(f"{out}__min"),
                pl.col(mean_mean_col).mean().alias(f"{out}__contract_mean_mean"),
                pl.col(mean_max_col).max().alias(f"{out}__contract_mean_max"),
                pl.col(std_mean_col).mean().alias(f"{out}__contract_std_mean"),
                pl.col(range_max_col).max().alias(f"{out}__contract_range_max"),
            ]
        )

    result = lf.group_by("case_id").agg(aggs).collect(engine="streaming")
    gc.collect()
    check_memory("after combine case partials")
    return result

In [ ]:
sample_submission = pd.read_csv(ROOT / "sample_submission.csv")
base = normalize_date_columns(pl.scan_parquet(str(TEST_DIR / "test_base.parquet"))).collect(engine="streaming")
case_ids = base.select("case_id").with_columns(pl.col("case_id").cast(pl.Int64))
print({
    "sample_submission_shape": sample_submission.shape,
    "test_base_shape": base.shape,
})
check_memory("after load base")

a2_paths = sorted(TEST_DIR.glob("test_credit_bureau_a_2_*.parquet"))
print({"a2_file_count": len(a2_paths), "a2_files": [path.name for path in a2_paths]})
if not a2_paths:
    raise FileNotFoundError("No test_credit_bureau_a_2_*.parquet files found.")

contract_dir = WORKING / "a2_contract_partials"
case_dir = WORKING / "a2_case_partials"
contract_dir.mkdir(parents=True, exist_ok=True)
case_dir.mkdir(parents=True, exist_ok=True)

case_paths = []
for idx, path in enumerate(a2_paths):
    contract_path = contract_dir / f"a2_contract_partial_{idx:02d}.parquet"
    case_path = case_dir / f"a2_case_partial_{idx:02d}.parquet"
    aggregate_file_to_contracts(path, case_ids, contract_path)
    case_paths.append(aggregate_contracts_to_case(contract_path, case_path))
    gc.collect()
    check_memory(f"after full file pipeline {path.name}")

a2_features = combine_case_partials(case_paths)
print({"a2_features_shape": a2_features.shape})

features = base.join(a2_features, on="case_id", how="left")
print({"final_feature_shape": features.shape})
check_memory("after final feature join")

feature_path = WORKING / "v6_a2_twostage_features.parquet"
features.write_parquet(feature_path)
print({"feature_path": str(feature_path)})

del a2_features, features, base, case_ids
gc.collect()
check_memory("after write final features")

In [ ]:
submission = sample_submission[["case_id"]].copy()
submission["score"] = 0.5
submission.to_csv(WORKING / "submission.csv", index=False)
submission.to_csv("submission.csv", index=False)

print({"submission_shape": submission.shape})
check_memory("submission done")
submission.head()